<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **MNPS Job Classification_gpt-4o_Qwen Corrected_(Two-Pass Self-Consistency)**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> # **Version 7.5.6 Changes**
>   - Explicitly mandates the LLM to use license and education information as decisive criteria.
>   -  It ensures the LLM's classification is final before we generate the job title.
> - **Two-Pass Classification**:
>   - Pass 1: Initial LLM classification (attribute-only, full context)
>   - Pass 2: Self-consistency check — LLM reviews its own justification vs classification and corrects mismatches
> - **Preserves all v7.5.3 logic**: Drive mounting, prompts, model, post-processing, validation
> - **Addresses Justification Mismatch Problem** without changing output format
> - **Still ignores original job title**, uses full MNPS role/competency context
> - **Same GPT-4o-2024-11-20 model**, rate-limiting, and output saving

In [ ]:
# ==== 1) Imports, paths, inputs from v7.1 artifacts ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# If main input file not found, check inside the zip
if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")

    # Optionally extract Ground Truth if present
    if "Ground Truth Masterfile.csv" in zip_contents:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)
            print("✅ Extracted Ground Truth Masterfile.csv from zip")

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

In [ ]:
# ==== 2) Load data and build attribute-only view (ignore title) ====
# Load prediction data (if available from previous runs)
# For now, we'll work with the raw job descriptions
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")

In [ ]:
# ==== 3) Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic ====
# Get MNPS roles from the loaded data
# Handle different possible column names
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    VALID_ROLES = roles_df[role_columns[0]].dropna().tolist()
else:
    # Fallback to first column
    VALID_ROLES = roles_df.iloc[:, 0].dropna().tolist()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")

# Enhanced closed sets for major and minor role groups
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator', 'Skilled Laborer',
    'Administrative Assistant'
]
MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

# Normalization mapping for minor roles
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Enhanced specialist fallback patterns with Problem Role Cheat Sheet logic
SPECIALIST_FALLBACKS = [
    # Technician patterns - hands-on technical work, equipment, maintenance
    ('Technician', 'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools|machinery|systems'),
    # Analyst patterns - data analysis, research, evaluation
    ('Analyst', 'analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports'),
    # Teacher patterns - classroom instruction, curriculum, students
    ('Teacher', 'classroom|lesson|instruction|teacher|students|curriculum|teaching|educational|academic'),
    # Coach patterns - mentoring, professional development, instructional support
    ('Coach', 'coach|instructional coach|plc|model lessons|co-teach|mentor|professional development|instructional support'),
    # Clerical Support patterns - administrative, office work, records
    ('Clerical Support', 'clerk|clerical|records|data entry|office support|administrative|filing|correspondence'),
    # Counselor patterns - guidance, therapy, mental health
    ('Counselor', 'counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological'),
    # Manager patterns - management, supervision, strategic planning
    ('Manager', 'manage|supervise|budget|oversight|lead team|program manager|direct|strategic|planning|policy'),
    # Accountant patterns - financial, accounting, bookkeeping
    ('Accountant', 'accounting|financial|bookkeeping|audit|budget|finance|accounts payable|accounts receivable|fiscal'),
    # Coordinator patterns - coordination, organization, facilitation
    ('Coordinator', 'coordinate|organize|facilitate|liaison|program coordination|project coordination|event coordination'),
    # Architect patterns - building/construction vs technology
    ('Architect (Facility-Focused)', 'building|construction|facility|architectural|design|space planning|renovation|infrastructure'),
    ('Architect (Technology-Focused)', 'system|software|technology|IT|database|network|programming|technical architecture')
]

# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    """Enhanced logic to discourage overuse of 'Specialist' based on Problem Role Cheat Sheet."""
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    # Check for more specific role matches first
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    # If no specific match, return Specialist
    return proposed_major

def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    """Distinguish between Supervisor and Manager based on education requirements.
    Supervisor: Primarily manages people, no post-high school education required
    Manager: Does more than manage people, requires minimum associates degree
    """
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Check for education requirements
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)
    # Check for broader responsibilities beyond people management
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)
    # If has degree requirement or broader responsibilities, likely Manager
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'
    # If primarily people management without degree requirements, likely Supervisor
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'
    return proposed_major

def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    """Refine distinctions between Coordinator, Coach, and Manager based on Problem Role Cheat Sheet."""
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    # Coach patterns - instructional support, mentoring, professional development
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'
    # Manager patterns - strategic planning, policy, budget, supervision
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'
    # Coordinator patterns - coordination, organization, facilitation
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'
    return proposed_major

def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    """Fix minor sub-grouping for executive roles - rarely "Lead", usually "I", "II", or "III"."""
    if major_role not in EXECUTIVE_ROLES:
        return minor_role
    # If it's an executive role and currently "Lead", downgrade to "III" or "II"
    if minor_role == 'Lead':
        # Check if it's a very senior executive role that might warrant "III"
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'
    return minor_role

print("✅ Enhanced closed sets and normalization helpers with Problem Role Cheat Sheet logic defined")

In [ ]:
# ==== 4) Build comprehensive KSACs text from all MNPS resources ====
def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n"
    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()

    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)
    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")

    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)
    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")

    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)
    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")

    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print("✅ Includes all 4 critical MNPS resource documents")

In [ ]:
zero_shot_prompt = \
"""Objective: Classify job descriptions into MNPS roles based SOLELY on job attributes, NOT job titles. Use the attached MNPS resources to ensure alignment.
Process:
- Analyze ALL provided attributes: Position Summary, Essential Functions, Work Experience, Education, Licenses and Certifications, Knowledge, Skills and Abilities.
- Classify based on the NATURE of the work, NOT the job title.
- CRITICAL: The 'Licenses and Certifications' field is a primary classification driver. If a job requires a nationally recognized, domain-specific professional license (e.g., Orientation and Mobility, Physical Therapist, Registered Behavior Technician), it is a 'Specialist', regardless of whether the job description uses words like 'teach' or 'instruct'. A license defines the role, not the title.
- CRITICAL: The 'Education' field helps determine complexity. A Master's Degree or higher, especially when paired with a professional license, typically indicates a 'III' level. An Associate's Degree or Bachelor's Degree typically supports a 'II' level. A High School Diploma or GED typically indicates an 'I' level.
- Use only approved MNPS roles and levels (I, II, III, Lead).

IMPORTANT CLASSIFICATION GUIDELINES (Based on Problem Role Cheat Sheet & MNPS Standards):
ROLE DISTINCTIONS:
- **Technician vs Specialist vs Analyst**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting. Requires no professional license.
  * Specialist: Requires a nationally recognized, domain-specific professional license (e.g., COMS, PT, RBT, PE, Architect) and/or advanced degree. Focuses on specialized expertise, not general instruction or data analysis.
  * Analyst: Data analysis, research, evaluation, assessment, statistical work, reporting. Uses data to inform decisions.
- **Coordinator vs Coach vs Manager**:
  * Coordinator: Coordination, organization, facilitation, liaison work, program coordination. Does NOT supervise staff or manage budgets.
  * Coach: Instructional support, mentoring, professional development, co-teaching, PLC facilitation. Focuses on improving teacher practice.
  * Manager: Strategic planning, policy development, budget oversight, direct supervision of staff, management of people and resources.
- **Supervisor vs Manager**:
  * Supervisor: Primarily manages people, no post-high school education required.
  * Manager: Does more than manage people, requires minimum associates degree and often a professional license.
- **Architect Roles**:
  * Architect (Facility-Focused): Building/construction/space planning/renovation/infrastructure. Requires a valid Architect license.
  * Architect (Technology-Focused): System/software/IT/database/network/programming. Requires advanced technical expertise.
MINOR SUB-GROUP GUIDELINES:
- **Executive Roles** (Coordinator, Principal, Director, Manager): Rarely "Lead", usually "I", "II", or "III". "Lead" is reserved for non-executive roles that lead teams or projects.
- **"III"** for senior-level expertise, advanced degrees (Master's+), and professional licenses.
- **"II"** for intermediate complexity and responsibility, typically requiring a Bachelor's degree.
- **"I"** for entry-level or basic complexity, typically requiring a High School Diploma or GED.
- **Teacher, Counselor, Librarian, Principal, Therapist, Social Worker**: These roles do NOT use I/II/III. Leave minor_sub_group BLANK. Only use "Lead" if the role meets specific senior leadership criteria.

Output Requirements:
- Major Role Group: Choose from approved MNPS major role groupings.
- Minor Sub Group: Use I, II, III, or Lead based on complexity and responsibility level. BLANK for Teacher, Counselor, Librarian, Principal, Therapist, Social Worker.
- new_job_title: Will be generated automatically after classification. DO NOT USE THIS FIELD TO INFER ROLE.
- Provide detailed justification based on job attributes and MNPS KSACs alignment that matches your selected role and level. Justify the role choice by explicitly referencing the required license or education level if applicable.""""

In [ ]:
# ==== 6) OpenAI API Setup with Rate Limiting Protection ====
import os
from google.colab import userdata

# Get API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

# Use GPT-4o-2024-11-20 for stable performance
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            error_str = str(e).lower()
            # Check for rate limiting errors
            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    # Exponential backoff with jitter
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting. Error: {e}")
                    raise e
            else:
                # Non-rate limiting error, raise immediately
                print(f"❌ Non-rate limiting error: {e}")
                raise e
    # This should never be reached, but just in case
    raise Exception("Unexpected error in retry logic")

print("✅ call_llm_json_with_retry function defined with rate limiting protection")

In [ ]:
# ==== 7) Two-Pass Batch Processing with Self-Consistency Check ====
from tqdm import tqdm

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description using a two-pass LLM approach with self-consistency check."""
    # Build job description text (ignore job title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    # === PASS 1: Initial Classification ===
    pass1_prompt = f"""{zero_shot_prompt}
Available MNPS Roles: {', '.join(VALID_ROLES)}
{KSACS_TEXT}
Job Description to Classify:
{job_text}
**IMPORTANT**:
- Ignore the job title completely
- Base classification solely on job attributes
- Use only approved MNPS roles and levels (I, II, III, Lead)
- Apply Problem Role Cheat Sheet guidelines
- Avoid overusing "Specialist"
- Distinguish Supervisor vs Manager based on education requirements
- Use Architect (Facility-Focused) for building/construction roles
- Use Architect (Technology-Focused) for system/software roles
- Executive roles (Coordinator, Principal, Director, Manager) rarely have "Lead" minor sub-grouping
- Ensure new_job_title incorporates both major_role_group and minor_sub_group
- Provide detailed justification that aligns with your selected role and level
Return your response as a JSON object with the following structure:
{{
  "new_job_title": "Descriptive title incorporating major_role_group and minor_sub_group",
  "major_role_group": "One of the approved MNPS roles",
  "minor_sub_group": "I, II, III, or Lead (consider executive role guidelines)",
  "grouping_justification": "Detailed explanation based on job attributes and KSACs alignment that matches your selected role and level"
}}"""

    try:
        # Get initial prediction
        pass1_response = call_llm_json_with_retry(pass1_prompt, MODEL_ID)

        # Apply post-processing logic (as in original)
        major_role = pass1_response.get('major_role_group', 'Other')
        minor_role = pass1_response.get('minor_sub_group', 'I')
        justification = pass1_response.get('grouping_justification', 'No justification provided')
        new_job_title = pass1_response.get('new_job_title', f"{major_role} {minor_role}")

        # Apply rule-based corrections
        major_role = discourage_specialist(job_text, major_role)
        major_role = distinguish_supervisor_manager(job_text, major_role)
        major_role = refine_coordinator_coach_manager(job_text, major_role)
        minor_role = normalize_minor(minor_role)
        minor_role = fix_executive_minor_sub_grouping(major_role, minor_role)
        if not new_job_title or new_job_title == 'Unknown':
            new_job_title = f"{major_role} {minor_role}"

        # Reconstruct clean Pass 1 output
        pass1_clean = {
            "new_job_title": new_job_title,
            "major_role_group": major_role,
            "minor_sub_group": minor_role,
            "grouping_justification": justification
        }

        # === PASS 2: Self-Consistency Check ===
        pass2_prompt = f"""{self_consistency_prompt}

**Your Previous Output:**
{json.dumps(pass1_clean, indent=2)}

**Now perform the self-consistency check and return the corrected (or unchanged) JSON.**
"""
        # Call LLM again for consistency check
        pass2_response = call_llm_json_with_retry(pass2_prompt, MODEL_ID)

        # Extract final values (do NOT re-apply post-processing to avoid overriding LLM correction)
        final_major = pass2_response.get('major_role_group', major_role)
        final_minor = pass2_response.get('minor_sub_group', minor_role)
        final_title = pass2_response.get('new_job_title', new_job_title)
        final_justification = pass2_response.get('grouping_justification', justification)  # should be unchanged

        # Re-normalize minor role (in case LLM outputs "1", etc.)
        final_minor = normalize_minor(final_minor)

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': final_title,
            'major_role_group': final_major,
            'minor_sub_group': final_minor,
            'grouping_justification': final_justification,
            'model_used': MODEL_ID
        }

    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions with rate limiting protection
results = []
print("🚀 Starting two-pass batch processing with self-consistency check and rate limiting protection...")
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    # Add small delay between requests to prevent rate limiting
    time.sleep(0.2)

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v754_two_pass.csv"
results_df.to_csv(output_path, index=False)
print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")

In [ ]:
# ==== 8) Generate Summary Statistics and Examples ====
# Load the results
preds = results_df.copy()

# Generate summary statistics
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

# Create summary
summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles', 'specialist_count', 'executive_lead_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum())
    ]
})
summary_path = OUTPUTS_DIR / "summary_stats_gpt4o_v754_two_pass.csv"
summary_stats.to_csv(summary_path, index=False)

# Show examples of classifications
examples = preds[['source_row_index', 'job_title_original', 'new_job_title',
                  'major_role_group', 'minor_sub_group']].head(10)
examples_path = OUTPUTS_DIR / "examples_gpt4o_v754_two_pass.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(summary_stats.to_string(index=False))
print("\n📝 Major Role Distribution:")
print(major_counts.to_string())
print("\n📝 Minor Role Distribution:")
print(minor_counts.to_string())
print("\n📝 Example Classifications:")
print(examples.to_string(index=False))
print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")

In [ ]:
# ==== 9) Enhanced Quality Check and Validation ====
# Check for alignment issues between justification and selected roles
alignment_issues = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()
    # Check if justification mentions the selected role
    if major_role not in justification and major_role != 'other':
        alignment_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': row['major_role_group'],
            'justification_excerpt': row['grouping_justification'][:100] + '...'
        })

# Check for job title format consistency
title_format_issues = []
for idx, row in preds.iterrows():
    new_title = str(row['new_job_title'])
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    # Check if job title incorporates both major and minor roles
    if major_role.lower() not in new_title.lower() or minor_role.lower() not in new_title.lower():
        title_format_issues.append({
            'row_index': row['source_row_index'],
            'new_job_title': new_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role
        })

# Check for executive roles with "Lead" minor sub-grouping (should be rare)
executive_lead_issues = []
for idx, row in preds.iterrows():
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    if major_role in EXECUTIVE_ROLES and minor_role == 'Lead':
        executive_lead_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'new_job_title': row['new_job_title']
        })

# Save quality check results
if alignment_issues:
    alignment_df = pd.DataFrame(alignment_issues)
    alignment_path = OUTPUTS_DIR / "alignment_issues_gpt4o_v754_two_pass.csv"
    alignment_df.to_csv(alignment_path, index=False)
    print(f"⚠️  Found {len(alignment_issues)} alignment issues - saved to {alignment_path}")
else:
    print("✅ No alignment issues found")

if title_format_issues:
    title_format_df = pd.DataFrame(title_format_issues)
    title_format_path = OUTPUTS_DIR / "title_format_issues_gpt4o_v754_two_pass.csv"
    title_format_df.to_csv(title_format_path, index=False)
    print(f"⚠️  Found {len(title_format_issues)} title format issues - saved to {title_format_path}")
else:
    print("✅ No title format issues found")

if executive_lead_issues:
    executive_lead_df = pd.DataFrame(executive_lead_issues)
    executive_lead_path = OUTPUTS_DIR / "executive_lead_issues_gpt4o_v754_two_pass.csv"
    executive_lead_df.to_csv(executive_lead_path, index=False)
    print(f"⚠️  Found {len(executive_lead_issues)} executive roles with 'Lead' minor sub-grouping - saved to {executive_lead_path}")
else:
    print("✅ No executive roles with inappropriate 'Lead' minor sub-grouping found")

print("\n✅ Enhanced quality check completed")